### **Structured output**

#### Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


### **Pydantic**
#### Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

model= ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature= 0.2
)
model

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.5-flash', temperature=0.2, client=<google.genai.client.Client object at 0x000001E5E71D1E80>, default_metadata=(), model_kwargs={})

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str= Field(description="The name of the movie")
    year:int= Field(description="The year in which the movie was released")
    rating:float= Field(description="The rating of the movie out of 10")
    director:str= Field(description="The director of the movie")

model_with_structure= model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.5-flash', temperature=0.2, client=<google.genai.client.Client object at 0x0000026D92024830>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'description': 'The name of the movie', 'title': 'Title', '

In [3]:
response= model_with_structure.invoke("Provide details about the moview Inception")
print(response)

title='Inception' year=2010 rating=8.8 director='Christopher Nolan'


### **Message output alongside parsed structure**

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details. """
    title: str = Field( ... , description="The title of the movie")             # ... => Ellipsis, empty argument
    year: int = Field( ... , description="The year the movie was released")
    director: str = Field( ... , description="The director of the movie")
    rating: float = Field( ... , description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', 'extras': {'signature': 'Eo4KCosKAQw51scfAhSqHJF0GpjJY6EuMAC5qhIy9NhMssD4UIXhsb0ekQfXmwDPLsBXiI3urtmS7YcYi05obNohB1qdFTJwzmTV4+mgkXv1RNZ14Sr+4jogltqzF5T6/bY05V8xniJKCHvpK8v8qveFJPGIKZiu+90RNEJh+pOoqtmzDCnd4TbAzaRHa6AK3D+LThX5hDL0TsZVSIJ4q40koyN/A3onnKKaO8d9w136hJ8HhR7S5oHKxyZYgEShaOnim/5BACaW0GqUnax2eG4xm0Wbdickm609Sn+iOElH0PEAlXJAfcHRlrRUf9dNuTl7/XdAnjoZ/e0sMfus1u0vrqnxX7OWgIoRrTbtWvEkUs+Q8pMw/Sg/I67IzCBRldqmaSDyjfWP1sPb2l8C63mLK7/ymQ8RfXcjAdBGTvffPWj1+s96eMhhI5jsAmrE3AN9+PJcpwRGql2VrPHEvoiegV24AyZDjsz9+bqkuDAOV657fN/2WNaZwK4fu3xwkp1o8hKaktVlXfkg4Y4X960C961O33kAYDr4cUw5BX72r7HmIuSm8ATO5GIQFIEoAgTcTCNbE8f13VVT1+fb2xBrGOOGRA4MHw3WhaG/4pWr+9SMmv+UAlSlG7/jitO0iQBbh6xHyFqaXF870B8an/qQQ/+MFpuIQHnPhTodjANUJIDd4V0rPMqDRn1q87l2tep60cniUGnuiWscz7aozUhwDjtitCnsUZ9q7e0WenU9az0swgursQ4qgvTNJOZLZCHf9yB9U2BeH+ncHSKLuZZVU/XESOYdVMK8E6qtIi5HuKqa99w/9EnBpQkdonI/vpu1qQCNU8

### **Nested Structure**: Multiple Classes

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str= Field(description="Name of the actor")
    role:str= Field(description="Role that the actor is playing")

class MovieDetails(BaseModel):
    title: str
    year:int
    cast: list[Actor]= Field(description="Information about actors", min_length=6)
    genres:list[str] 
    budget: float | None= Field(None, description="Budget in millions USD")

model_with_structure= model.with_structured_output(MovieDetails)
response= model_with_structure.invoke("Provide details about the movie Interstellar.")
response

MovieDetails(title='Interstellar', year=2014, cast=[Actor(name='Matthew McConaughey', role='Cooper'), Actor(name='Anne Hathaway', role='Brand'), Actor(name='Jessica Chastain', role='Murph'), Actor(name='Bill Irwin', role='TARS'), Actor(name='Ellen Burstyn', role='Murph (older)'), Actor(name='Michael Caine', role='Professor Brand')], genres=['Adventure', 'Drama', 'Sci-Fi'], budget=165.0)

### **TypedDict**

#### TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need "Runtime validation".

In [12]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name: Annotated[str, ..., "Name of the actor"]
    role: Annotated[str, ..., "Role of the actor"]
    age: Annotated[int, ..., "Age of the actor/actress"]

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of release of the movie"]
    cast: Annotated[list[Actor], ..., "The actors in the movie"]
    director: Annotated[str, ..., "The name of the director"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typedict= model.with_structured_output(MovieDict)
response= model_with_typedict.invoke("Provide details about the movie Oppenheimer")
response

{'title': 'Oppenheimer',
 'year': 2023,
 'cast': [{'name': 'Cillian Murphy',
   'role': 'J. Robert Oppenheimer',
   'age': 47},
  {'name': 'Emily Blunt', 'role': "Katherine 'Kitty' Oppenheimer", 'age': 40},
  {'name': 'Matt Damon', 'role': 'Leslie Groves', 'age': 53},
  {'name': 'Robert Downey Jr.', 'role': 'Lewis Strauss', 'age': 58},
  {'name': 'Florence Pugh', 'role': 'Jean Tatlock', 'age': 27}],
 'director': 'Christopher Nolan',
 'rating': 8.4}

### create_agent using Pydantic, TypedDict and Dataclass

In [ ]:
# Using Pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    response_format= ContactInfo         # Auto-selects ProviderStrategy
)

result = agent. invoke(
    {
        "messages": [
            {  
                "role": "user", 
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)
print(result)
# print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone=' (555) 123-4567')

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='24d95f0a-228f-483a-a3a4-2b60d07955d4'), AIMessage(content=[{'type': 'text', 'text': '{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', 'extras': {'signature': 'EjQKMgEMOdbHFOFCWbK3hfZ/ilXPd5uCkwHwWe4CBFd4mmP6ILkkHaAbHGeJ7JaxpPoGNTtW'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7567-94a3-7971-b563-48650a891d48-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})], 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}
name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [ ]:
# Using TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str                      # The name of the person
    email: str                     # The email address of the person
    phone: str                     # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    response_format= ContactInfo        # Auto-selects ProviderStrategy
)

result = agent. invoke(
    {
        "messages": [
            {
                "role": "user", 
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

### **DataClasses**
#### A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [ ]:
# Using Dataclass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person. """
    name: str               # The name of the person
    email: str              # The email address of the person
    phone: str              # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    response_format= ContactInfo
)
# Auto-selects ProviderStrategy

result = agent. invoke(
    {
        "messages": 
        [
            {
                "role": "user", 
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [2]:
model.profile

{'name': 'Gemini 3.5 Flash',
 'release_date': '2026-05-19',
 'last_updated': '2026-05-19',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}